# Split CIFAR-10: does the bracket still beat EWC on real photos?

Same tests as MNIST, in one pass per run, on colour photos with a small CNN:
- 5 tasks: airplane vs automobile, bird vs cat, deer vs dog, frog vs horse, ship vs truck
- **No old data:** bracket from the stored map, EWC penalty, estimated watching
- **Tiny buffer:** bracket and simple check with 10 and 50 old images
- **References (use old data):** full bracket, simple check

Kernel: `E:\wordstest\.venv\Scripts\python.exe`. Run top to bottom. Press **Ctrl+S** after each run.

## Cell 1: settings
**Should print:** the folders. CIFAR is slower than MNIST. If seed 0 takes more than about 4 minutes, set `SEEDS = 3`, rerun this cell and Cell 4, then continue.

In [1]:
import os
BASE = r"F:\forgetting"
DATA_DIR = os.path.join(BASE, "data")
RESULTS_DIR = os.path.join(BASE, "results_cifar")
os.makedirs(RESULTS_DIR, exist_ok=True)
SEEDS = 5
EPOCHS = 3
LR = 0.05
BATCH = 64
EST_SIZE = 2000
K = 20
BUF = (10, 50)
print("data:", DATA_DIR); print("results:", RESULTS_DIR)

data: F:\forgetting\data
results: F:\forgetting\results_cifar


## Cell 2: GPU + libraries
**Should print:** `cuda` and the 3090.

In [2]:
import time, json, csv
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.func import functional_call, vmap, grad as fgrad
from scipy.stats import spearmanr
from scipy.sparse.linalg import LinearOperator, eigsh
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", dev)
if dev.type == "cuda": print(torch.cuda.get_device_name(0))

torch 2.5.1+cu121 | device: cuda
NVIDIA GeForce RTX 3090


## Cell 3: load CIFAR-10 (from your drive)
**Should print:** `train 50000 test 10000`, then 5 tasks with 10,000 train / 2,000 test each.

In [3]:
from torchvision import datasets
tr = datasets.CIFAR10(DATA_DIR, train=True, download=True)
te = datasets.CIFAR10(DATA_DIR, train=False, download=True)
MEAN = torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1); STD = torch.tensor([0.2470, 0.2435, 0.2616]).view(1, 3, 1, 1)
def prep(ds):
    X = torch.tensor(ds.data).float().div(255.).permute(0, 3, 1, 2)
    return (X - MEAN) / STD, torch.tensor(ds.targets)
Xtr_all, ytr_all = prep(tr); Xte_all, yte_all = prep(te)
print("train", len(ytr_all), "test", len(yte_all))
TASKS = [(0, 1), (2, 3), (4, 5), (6, 7), (8, 9)]
NAMES = ["airplane vs automobile", "bird vs cat", "deer vs dog", "frog vs horse", "ship vs truck"]
data = {}
g0 = torch.Generator().manual_seed(123)
for t, cls in enumerate(TASKS):
    mtr = (ytr_all == cls[0]) | (ytr_all == cls[1]); mte = (yte_all == cls[0]) | (yte_all == cls[1])
    Xtr, ytr, Xte, yte = Xtr_all[mtr], ytr_all[mtr], Xte_all[mte], yte_all[mte]
    est = torch.randperm(len(ytr), generator=g0)[:EST_SIZE]
    data[t] = dict(Xtr=Xtr.to(dev), ytr=ytr.to(dev), Xte=Xte.to(dev), yte=yte.to(dev),
                   Xest=Xtr[est].to(dev), yest=ytr[est].to(dev), cls=torch.tensor(cls, device=dev))
    print(f"task {t} ({NAMES[t]}): train {len(ytr)}, test {len(yte)}")

Files already downloaded and verified
Files already downloaded and verified
train 50000 test 10000
task 0 (airplane vs automobile): train 10000, test 2000
task 1 (bird vs cat): train 10000, test 2000
task 2 (deer vs dog): train 10000, test 2000
task 3 (frog vs horse): train 10000, test 2000
task 4 (ship vs truck): train 10000, test 2000


## Cell 4: small CNN and the maths
Two conv layers (tanh, average pooling, so everything stays smooth) and a small dense head.

**Should print:** `ready` and the number of parameters.

In [4]:
SCENARIO = "task"
class CNN(nn.Module):
    def __init__(self, out_dim):
        super().__init__()
        self.c1 = nn.Conv2d(3, 16, 3, padding=1); self.c2 = nn.Conv2d(16, 32, 3, padding=1)
        self.f1 = nn.Linear(32 * 8 * 8, 64); self.f2 = nn.Linear(64, out_dim)
    def forward(self, x):
        x = F.avg_pool2d(torch.tanh(self.c1(x)), 2)
        x = F.avg_pool2d(torch.tanh(self.c2(x)), 2)
        return self.f2(torch.tanh(self.f1(x.flatten(1))))

def setup(scenario):
    global SCENARIO, model, P_NAMES, P_SHAPES, P_SIZES, NPARAM
    SCENARIO = scenario
    model = CNN(2 if scenario == "domain" else 10).to(dev)
    P_NAMES = [n for n, _ in model.named_parameters()]
    P_SHAPES = [p.shape for _, p in model.named_parameters()]
    P_SIZES = [p.numel() for _, p in model.named_parameters()]
    NPARAM = sum(P_SIZES)

def unflat(th):
    out, i = {}, 0
    for n, s, k in zip(P_NAMES, P_SHAPES, P_SIZES):
        out[n] = th[i:i + k].view(s); i += k
    return out

def init(seed):
    torch.manual_seed(seed)
    m = CNN(2 if SCENARIO == "domain" else 10).to(dev)
    return torch.cat([p.detach().flatten() for p in m.parameters()])

def logits(th, X, cls):
    z = functional_call(model, unflat(th), (X,))
    return z if SCENARIO == "domain" else z[:, cls]

def task_loss(th, X, y, cls):
    return F.cross_entropy(logits(th, X, cls), (y == cls[1]).long())

def grad_on(th, X, y, cls):
    th = th.detach().requires_grad_(True)
    return torch.autograd.grad(task_loss(th, X, y, cls), th)[0].detach()

def hvp_on(th, X, y, cls, v):
    th = th.detach().requires_grad_(True)
    g = torch.autograd.grad(task_loss(th, X, y, cls), th, create_graph=True)[0]
    return torch.autograd.grad(g @ v.detach(), th)[0].detach()

def grad(th, t, X=None, y=None):
    d = data[t]
    return grad_on(th, d["Xest"] if X is None else X, d["yest"] if y is None else y, d["cls"])

def hvp(th, t, v):
    d = data[t]; return hvp_on(th, d["Xest"], d["yest"], d["cls"], v)

@torch.no_grad()
def loss_on(th, X, y, cls):
    return float(task_loss(th, X, y, cls))

@torch.no_grad()
def evaluate(th, t, split="te"):
    d = data[t]; X, y = d["X" + split], d["y" + split]
    z = logits(th, X, d["cls"]); yi = (y == d["cls"][1]).long()
    return float((z.argmax(1) == yi).float().mean()), float(F.cross_entropy(z, yi))

def train(th, t, seed, record=None, record_every=5):
    d = data[t]; th = th.detach().clone()
    gen = torch.Generator().manual_seed(seed * 1000 + t)
    n = len(d["ytr"]); step = 0
    for ep in range(EPOCHS):
        idx = torch.randperm(n, generator=gen).to(dev)
        for i in range(0, n, BATCH):
            bi = idx[i:i + BATCH]
            if record is not None and step % record_every == 0: record(step, th)
            th = th - LR * grad(th, t, d["Xtr"][bi], d["ytr"][bi])
            step += 1
    if record is not None: record(step, th)
    return th

def top_hessian(th, t, k=K):
    def mv(v):
        v_t = torch.as_tensor(np.asarray(v).ravel(), dtype=torch.float32, device=dev)
        return hvp(th, t, v_t).double().cpu().numpy()
    vals, vecs = eigsh(LinearOperator((NPARAM, NPARAM), matvec=mv, dtype=np.float64), k=k, which="LA", tol=1e-3)
    return torch.tensor(vals, dtype=torch.float32, device=dev), torch.tensor(vecs, dtype=torch.float32, device=dev)

def diag_fisher(th, t, chunk=100):
    d = data[t]; cls = d["cls"]
    per = vmap(fgrad(lambda th_, x, y: task_loss(th_, x[None], y[None], cls)), in_dims=(None, 0, 0))
    Fd = torch.zeros(NPARAM, device=dev)
    for i in range(0, len(d["yest"]), chunk):
        Fd += (per(th.detach(), d["Xest"][i:i + chunk], d["yest"][i:i + chunk]) ** 2).sum(0)
    return Fd / len(d["yest"])

def make_summary(th, t):
    lam, U = top_hessian(th, t)
    return dict(theta=th.detach().clone(), g=grad(th, t), lam=lam, U=U, F=diag_fisher(th, t))

def H_stored(S, v):
    return S["U"] @ (S["lam"] * (S["U"].T @ v))

setup("task"); print("ready | parameters:", NPARAM)

ready | parameters: 136874


## Cell 5: quick checks (a couple of minutes)
1. **Symmetry:** two nearly equal numbers.
2. **Accuracy:** one task trained alone. Photos are harder than digits, so **above about 0.80** is fine.
3. **Formula check:** ratio close to 1.00 at small learning rates.
4. **Stored map:** direction match at the switch, ideally above 0.9.

**Send Claude this output before running Cell 7.**

In [5]:
setup("domain")
th = init(99)
u, v = torch.randn(NPARAM, device=dev), torch.randn(NPARAM, device=dev)
print(f"1. symmetry: {float(u @ hvp(th, 0, v)):.5f}  vs  {float(v @ hvp(th, 0, u)):.5f}")
t0 = time.time(); thA = train(init(0), 0, 0)
print(f"2. task 0 accuracy: {evaluate(thA, 0)[0]:.4f}  (one task took {time.time() - t0:.1f}s)")
th = init(0); ga, gb = grad(th, 0), grad(th, 1); br = hvp(th, 1, ga) - hvp(th, 0, gb)
for eta in [0.1, 0.03, 0.01, 0.003]:
    t1 = th - eta * ga; tAB = t1 - eta * grad(t1, 1)
    t2 = th - eta * gb; tBA = t2 - eta * grad(t2, 0)
    d = tAB - tBA; pr = eta ** 2 * br
    print(f"3. lr {eta:<6} ratio {float(d.norm() / pr.norm()):.3f}  direction match {float(F.cosine_similarity(d, pr, dim=0)):.3f}")
t0 = time.time(); S = make_summary(thA, 0)
gA, gB = grad(thA, 0), grad(thA, 1)
full = hvp(thA, 1, gA) - hvp(thA, 0, gB); st = hvp(thA, 1, S["g"]) - H_stored(S, gB)
print(f"4. summary built in {time.time() - t0:.1f}s | stored vs full at the switch: direction {float(F.cosine_similarity(full, st, dim=0)):.3f}, size {float(st.norm() / full.norm()):.3f}")

1. symmetry: 15.82368  vs  15.82754
2. task 0 accuracy: 0.8460  (one task took 2.5s)
3. lr 0.1    ratio 1.005  direction match 0.933
3. lr 0.03   ratio 0.992  direction match 0.994
3. lr 0.01   ratio 0.996  direction match 0.999
3. lr 0.003  ratio 0.999  direction match 1.000
4. summary built in 5.4s | stored vs full at the switch: direction 1.000, size 1.000


## Cell 6: the experiment function (everything in one pass)
**Should print:** `defined`.

In [6]:
KEYS = ["bracket_stored", "ewc_penalty", "est_watch", "bracket_full", "simple_check"] + \
       [f"bracket_buf{n}" for n in BUF] + [f"check_buf{n}" for n in BUF]
LABELS = {"bracket_stored": "Bracket, stored map (no data)", "ewc_penalty": "EWC penalty (no data)",
          "est_watch": "Estimated watching (no data)", "bracket_buf10": "Bracket, 10 images", "check_buf10": "Simple check, 10 images",
          "bracket_buf50": "Bracket, 50 images", "check_buf50": "Simple check, 50 images",
          "bracket_full": "Ref: full bracket (old data)", "simple_check": "Ref: simple check (old data)"}

def run(scenario, lr=0.05, seeds=SEEDS):
    global LR
    LR = lr; setup(scenario)
    pairs = [(a, b) for a in range(5) for b in range(5) if a != b]
    vals = {k: [] for k in KEYS}; target, forget, accs = [], [], []; t0 = time.time()
    for seed in range(seeds):
        th0 = init(seed)
        th_task = {t: train(th0, t, seed) for t in range(5)}
        base = {t: evaluate(th_task[t], t) for t in range(5)}
        base_est = {t: evaluate(th_task[t], t, "est")[1] for t in range(5)}
        summ = {t: make_summary(th_task[t], t) for t in range(5)}
        buf = {}
        for t in range(5):
            d = data[t]; g = torch.Generator().manual_seed(seed * 77 + t)
            idx = torch.randperm(len(d["ytr"]), generator=g)[:max(BUF)].to(dev)
            buf[t] = {n: (d["Xtr"][idx[:n]], d["ytr"][idx[:n]]) for n in BUF}
        accs += [base[t][0] for t in range(5)]
        for a, b in pairs:
            S = summ[a]; cA = data[a]["cls"]; log = []
            base_buf = {n: loss_on(th_task[a], *buf[a][n], cA) for n in BUF}
            def rec(step, th_, S=S, a=a, b=b, cA=cA, log=log, base_buf=base_buf):
                if step not in (5, 10, 15, 20): return
                dth = th_ - S["theta"]; gB = grad(th_, b)
                gA_hat = S["g"] + H_stored(S, dth); Ud = S["U"].T @ dth
                row = {"bracket_stored": float((hvp(th_, b, gA_hat) - H_stored(S, gB)).norm()),
                       "ewc_penalty": float((S["F"] * dth ** 2).sum()),
                       "est_watch": float(S["g"] @ dth + 0.5 * (S["lam"] * Ud ** 2).sum()),
                       "bracket_full": float((hvp(th_, b, grad(th_, a)) - hvp(th_, a, gB)).norm()),
                       "simple_check": evaluate(th_, a, "est")[1] - base_est[a]}
                for n in BUF:
                    Xb, yb = buf[a][n]
                    row[f"bracket_buf{n}"] = float((hvp(th_, b, grad_on(th_, Xb, yb, cA)) - hvp_on(th_, Xb, yb, cA, gB)).norm())
                    row[f"check_buf{n}"] = loss_on(th_, Xb, yb, cA) - base_buf[n]
                log.append(row)
            thAB = train(th_task[a], b, seed, record=rec)
            accA1, lossA1 = evaluate(thAB, a)
            target.append(lossA1 - base[a][1]); forget.append(base[a][0] - accA1)
            for k in KEYS: vals[k].append(float(np.mean([r[k] for r in log])))
        print(f"  seed {seed} done after {time.time() - t0:.0f}s | mean task accuracy {np.mean([base[t][0] for t in range(5)]):.4f}")
    rho = {k: float(spearmanr(vals[k], target)[0]) for k in KEYS}
    Sm = {"scenario": scenario, "lr": lr, "seeds": seeds, "mean_task_accuracy": float(np.mean(accs)),
          "forgetting_mean": float(np.mean(forget)), "forgetting_max": float(np.max(forget)), "rho": rho}
    json.dump(Sm, open(os.path.join(RESULTS_DIR, f"cifar_{scenario}_lr{lr}.json"), "w"), indent=2)
    with open(os.path.join(RESULTS_DIR, f"cifar_{scenario}_lr{lr}.csv"), "w", newline="") as f:
        w = csv.writer(f); w.writerow(["loss_increase", "forgetting"] + KEYS)
        for i in range(len(target)): w.writerow([target[i], forget[i]] + [vals[k][i] for k in KEYS])
    print(f"\n== CIFAR-10 | {'slow' if scenario == 'task' else 'fast'} ({scenario}) | lr {lr} | task acc {Sm['mean_task_accuracy']:.3f} | "
          f"forgetting mean {Sm['forgetting_mean']:.3f}, max {Sm['forgetting_max']:.3f}")
    for k in ["bracket_stored", "ewc_penalty", "est_watch", "bracket_buf10", "check_buf10", "bracket_buf50", "check_buf50", "bracket_full", "simple_check"]:
        print(f"  {LABELS[k]:34s} {rho[k]:6.2f}")
    return Sm
print("defined")

defined


## Cell 7: main run, lr 0.05, both settings
Send Claude both tables.

In [7]:
CIF = {}
for sc in ["task", "domain"]:
    CIF[(sc, 0.05)] = run(sc, lr=0.05)

  seed 0 done after 107s | mean task accuracy 0.7792
  seed 1 done after 213s | mean task accuracy 0.7736
  seed 2 done after 316s | mean task accuracy 0.7861
  seed 3 done after 421s | mean task accuracy 0.7862
  seed 4 done after 528s | mean task accuracy 0.7568

== CIFAR-10 | slow (task) | lr 0.05 | task acc 0.776 | forgetting mean 0.041, max 0.141
  Bracket, stored map (no data)       -0.12
  EWC penalty (no data)                0.22
  Estimated watching (no data)         0.32
  Bracket, 10 images                  -0.12
  Simple check, 10 images              0.11
  Bracket, 50 images                  -0.21
  Simple check, 50 images              0.19
  Ref: full bracket (old data)        -0.11
  Ref: simple check (old data)         0.31
  seed 0 done after 101s | mean task accuracy 0.7789
  seed 1 done after 200s | mean task accuracy 0.7732
  seed 2 done after 296s | mean task accuracy 0.7853
  seed 3 done after 395s | mean task accuracy 0.7871
  seed 4 done after 497s | mean task a

In [8]:
def run_diag(scenario, lr=0.05, epochs=3, seeds=3):
    global LR, EPOCHS
    LR, EPOCHS = lr, epochs; setup(scenario)
    conv = torch.zeros(NPARAM, dtype=torch.bool, device=dev); i = 0
    for n, k in zip(P_NAMES, P_SIZES):
        if n.startswith("c"): conv[i:i + k] = True
        i += k
    DK = ["bracket_full", "br_conv", "br_head", "br_onA_full", "br_onA_stored", "br_topA_stored",
          "ewc_penalty", "est_watch", "simple_check"]
    pairs = [(a, b) for a in range(5) for b in range(5) if a != b]
    vals = {k: [] for k in DK}; target, forget, accs = [], [], []; t0 = time.time()
    for seed in range(seeds):
        th0 = init(seed); th_task = {t: train(th0, t, seed) for t in range(5)}
        base = {t: evaluate(th_task[t], t) for t in range(5)}
        base_est = {t: evaluate(th_task[t], t, "est")[1] for t in range(5)}
        summ = {t: make_summary(th_task[t], t) for t in range(5)}
        accs += [base[t][0] for t in range(5)]
        for a, b in pairs:
            S = summ[a]; log = []
            def rec(step, th_, S=S, a=a, b=b, log=log):
                if step not in (5, 10, 15, 20): return
                dth = th_ - S["theta"]; gA = grad(th_, a); gB = grad(th_, b)
                bf = hvp(th_, b, gA) - hvp(th_, a, gB)
                gA_hat = S["g"] + H_stored(S, dth)
                bs = hvp(th_, b, gA_hat) - H_stored(S, gB)
                Ud = S["U"].T @ dth
                log.append({"bracket_full": float(bf.norm()),
                            "br_conv": float(bf[conv].norm()), "br_head": float(bf[~conv].norm()),
                            "br_onA_full": float(gA @ bf), "br_onA_stored": float(gA_hat @ bs),
                            "br_topA_stored": float((S["U"].T @ bs).norm()),
                            "ewc_penalty": float((S["F"] * dth ** 2).sum()),
                            "est_watch": float(S["g"] @ dth + 0.5 * (S["lam"] * Ud ** 2).sum()),
                            "simple_check": evaluate(th_, a, "est")[1] - base_est[a]})
            thAB = train(th_task[a], b, seed, record=rec)
            accA1, lossA1 = evaluate(thAB, a)
            target.append(lossA1 - base[a][1]); forget.append(base[a][0] - accA1)
            for k in DK: vals[k].append(float(np.mean([r[k] for r in log])))
        print(f"  seed {seed} done after {time.time() - t0:.0f}s | mean task accuracy {np.mean([base[t][0] for t in range(5)]):.4f}")
    rho = {k: float(spearmanr(vals[k], target)[0]) for k in DK}
    Sm = {"scenario": scenario, "lr": lr, "epochs": epochs, "seeds": seeds, "mean_task_accuracy": float(np.mean(accs)),
          "forgetting_mean": float(np.mean(forget)), "forgetting_max": float(np.max(forget)), "rho": rho}
    json.dump(Sm, open(os.path.join(RESULTS_DIR, f"diag_{scenario}_lr{lr}_ep{epochs}.json"), "w"), indent=2)
    print(f"\n== DIAG | {'slow' if scenario == 'task' else 'fast'} | lr {lr} | epochs {epochs} | task acc {Sm['mean_task_accuracy']:.3f} | forgetting mean {Sm['forgetting_mean']:.3f}")
    for k in DK: print(f"  {k:16s} {rho[k]:6.2f}")
    return Sm

for ep in [3, 6]:
    for sc in ["task", "domain"]:
        run_diag(sc, lr=0.05, epochs=ep)
EPOCHS = 3

  seed 0 done after 101s | mean task accuracy 0.7792
  seed 1 done after 203s | mean task accuracy 0.7736
  seed 2 done after 300s | mean task accuracy 0.7861

== DIAG | slow | lr 0.05 | epochs 3 | task acc 0.780 | forgetting mean 0.044
  bracket_full       0.04
  br_conv           -0.00
  br_head            0.05
  br_onA_full        0.09
  br_onA_stored      0.02
  br_topA_stored    -0.01
  ewc_penalty        0.26
  est_watch          0.43
  simple_check       0.43
  seed 0 done after 97s | mean task accuracy 0.7789
  seed 1 done after 192s | mean task accuracy 0.7732
  seed 2 done after 284s | mean task accuracy 0.7853

== DIAG | fast | lr 0.05 | epochs 3 | task acc 0.779 | forgetting mean 0.220
  bracket_full       0.30
  br_conv            0.25
  br_head            0.36
  br_onA_full        0.41
  br_onA_stored      0.49
  br_topA_stored     0.36
  ewc_penalty        0.54
  est_watch          0.73
  simple_check       0.75
  seed 0 done after 160s | mean task accuracy 0.8077
  seed

## Cell 8: learning-rate sweep (0.02 and 0.1)
About twice as long as Cell 7. Edit the list if time is short.

In [9]:
for lr in [0.02]:
    for sc in ["task", "domain"]:
        CIF[(sc, lr)] = run(sc, lr=lr)

  seed 0 done after 103s | mean task accuracy 0.7752
  seed 1 done after 211s | mean task accuracy 0.7730
  seed 2 done after 314s | mean task accuracy 0.7745
  seed 3 done after 420s | mean task accuracy 0.7803
  seed 4 done after 519s | mean task accuracy 0.7762

== CIFAR-10 | slow (task) | lr 0.02 | task acc 0.776 | forgetting mean 0.019, max 0.087
  Bracket, stored map (no data)        0.22
  EWC penalty (no data)                0.43
  Estimated watching (no data)         0.18
  Bracket, 10 images                   0.08
  Simple check, 10 images              0.06
  Bracket, 50 images                   0.17
  Simple check, 50 images              0.16
  Ref: full bracket (old data)         0.22
  Ref: simple check (old data)         0.18
  seed 0 done after 92s | mean task accuracy 0.7750
  seed 1 done after 188s | mean task accuracy 0.7717
  seed 2 done after 281s | mean task accuracy 0.7753
  seed 3 done after 381s | mean task accuracy 0.7793
  seed 4 done after 476s | mean task ac

In [10]:
DS = "cifar" if "cifar" in RESULTS_DIR else "mnist"

def run_order(scenario, lr=0.05, seeds=3):
    global LR
    LR = lr; setup(scenario)
    pairs = [(a, b) for a in range(5) for b in range(5) if a != b]
    K_ = ["br_start", "br_onA_start", "br_early", "forgetting_loss"]
    vals = {k: [] for k in K_}; gapA, gapP = [], []; t0 = time.time()
    for seed in range(seeds):
        th0 = init(seed)
        th_task = {t: train(th0, t, seed) for t in range(5)}
        base = {t: evaluate(th_task[t], t)[1] for t in range(5)}
        g0 = {t: grad(th0, t) for t in range(5)}
        thAB, early = {}, {}
        for a, b in pairs:
            log = []
            def rec(step, th_, a=a, b=b, log=log):
                if step in (5, 10, 15, 20):
                    log.append(float((hvp(th_, b, grad(th_, a)) - hvp(th_, a, grad(th_, b))).norm()))
            thAB[(a, b)] = train(th_task[a], b, seed, record=rec)
            early[(a, b)] = float(np.mean(log))
        for a, b in pairs:
            br0 = hvp(th0, b, g0[a]) - hvp(th0, a, g0[b])
            lossA_AB = evaluate(thAB[(a, b)], a)[1]          # A learned first, then B
            lossA_BA = evaluate(thAB[(b, a)], a)[1]          # A learned last
            gapA.append(lossA_AB - lossA_BA)
            gapP.append(float((thAB[(a, b)] - thAB[(b, a)]).norm()))
            vals["br_start"].append(float(br0.norm()))
            vals["br_onA_start"].append(float(g0[a] @ br0))
            vals["br_early"].append(early[(a, b)])
            vals["forgetting_loss"].append(lossA_AB - base[a])
        print(f"  seed {seed} done after {time.time() - t0:.0f}s")
    r = lambda x, y: float(spearmanr(x, y)[0])
    out = {
        "order_gap_on_A vs br_start": r(vals["br_start"], gapA),
        "order_gap_on_A vs br_onA_start": r(vals["br_onA_start"], gapA),
        "order_gap_on_A vs br_early": r(vals["br_early"], gapA),
        "weight_gap vs br_start": r(vals["br_start"], gapP),
        "weight_gap vs br_early": r(vals["br_early"], gapP),
        "forgetting vs order_gap_on_A": r(gapA, vals["forgetting_loss"]),
        "forgetting vs br_early": r(vals["br_early"], vals["forgetting_loss"]),
    }
    share = float(np.mean(np.abs(gapA)) / np.mean(np.abs(vals["forgetting_loss"])))
    Sm = {"dataset": DS, "scenario": scenario, "lr": lr, "seeds": seeds, "rho": out, "order_share_of_forgetting": share}
    json.dump(Sm, open(os.path.join(RESULTS_DIR, f"order_{DS}_{scenario}_lr{lr}.json"), "w"), indent=2)
    print(f"\n== ORDER | {DS} | {'slow' if scenario == 'task' else 'fast'} | lr {lr}")
    for k, v in out.items(): print(f"  {k:34s} {v:6.2f}")
    print(f"  order effect as share of forgetting  {share:6.2f}")
    return Sm

for sc in ["task", "domain"]:
    run_order(sc, lr=0.05)

  seed 0 done after 71s
  seed 1 done after 144s
  seed 2 done after 218s

== ORDER | cifar | slow | lr 0.05
  order_gap_on_A vs br_start           0.33
  order_gap_on_A vs br_onA_start       0.00
  order_gap_on_A vs br_early           0.03
  weight_gap vs br_start               0.56
  weight_gap vs br_early               0.16
  forgetting vs order_gap_on_A         0.96
  forgetting vs br_early               0.04
  order effect as share of forgetting    1.20
  seed 0 done after 67s
  seed 1 done after 134s
  seed 2 done after 201s

== ORDER | cifar | fast | lr 0.05
  order_gap_on_A vs br_start           0.26
  order_gap_on_A vs br_onA_start       0.44
  order_gap_on_A vs br_early           0.29
  weight_gap vs br_start               0.49
  weight_gap vs br_early               0.39
  forgetting vs order_gap_on_A         0.99
  forgetting vs br_early               0.30
  order effect as share of forgetting    1.05


## Cell 9: summary table + headline chart (reads from the drive)

In [11]:
import glob
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams.update({"font.family": "Calibri", "font.size": 12, "axes.spines.top": False, "axes.spines.right": False})
R = {}
for f in glob.glob(os.path.join(RESULTS_DIR, "cifar_*.json")):
    d = json.load(open(f)); R[(d["scenario"], d["lr"])] = d
LRS = sorted({lr for _, lr in R})
cols = ["ewc_penalty", "bracket_stored", "bracket_buf10", "check_buf10", "bracket_buf50", "check_buf50", "bracket_full", "simple_check"]
heads = ["EWC", "Br stored", "Br 10", "Chk 10", "Br 50", "Chk 50", "Ref full br", "Ref check"]
print(f"{'setting':16s}" + "".join(f"{h:>12s}" for h in heads))
for sc in ["task", "domain"]:
    for lr in LRS:
        if (sc, lr) in R:
            r = R[(sc, lr)]["rho"]
            print(f"{('slow' if sc == 'task' else 'fast') + ' lr ' + str(lr):16s}" + "".join(f"{r[c]:12.2f}" for c in cols))
labels, br, ew = [], [], []
for sc, key, tag in [("task", "bracket_stored", "Slow, no data"), ("domain", "bracket_buf10", "Fast, 10 images")]:
    for lr in LRS:
        if (sc, lr) in R:
            labels.append(f"{tag}\nlr {lr}"); br.append(R[(sc, lr)]["rho"][key]); ew.append(R[(sc, lr)]["rho"]["ewc_penalty"])
fig, ax = plt.subplots(figsize=(9, 4.6), dpi=200); x = np.arange(len(labels)); w = 0.38
ax.bar(x - w / 2, br, w, color="#1F8A70", label="Lie bracket"); ax.bar(x + w / 2, ew, w, color="#B8651C", label="EWC (no data)")
for xi, v in zip(x - w / 2, br): ax.text(xi, v + 0.015, f"{v:.2f}", ha="center", fontsize=10)
for xi, v in zip(x + w / 2, ew): ax.text(xi, v + 0.015, f"{v:.2f}", ha="center", fontsize=10)
ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=10); ax.set_ylim(min(0, min(br + ew) - 0.05), 1)
ax.set_ylabel("Rank correlation with forgetting"); ax.grid(axis="y", color="#E3E8E6"); ax.set_axisbelow(True)
ax.legend(frameon=False, loc="upper right")
ax.set_title("CIFAR-10: Lie bracket vs EWC at predicting forgetting", loc="left", weight="bold", fontsize=12)
fig.tight_layout(); fig.savefig(os.path.join(RESULTS_DIR, "cifar_headline.png"), facecolor="white"); plt.close(fig)
print("saved", os.path.join(RESULTS_DIR, "cifar_headline.png"))

setting                  EWC   Br stored       Br 10      Chk 10       Br 50      Chk 50 Ref full br   Ref check
slow lr 0.02            0.43        0.22        0.08        0.06        0.17        0.16        0.22        0.18
slow lr 0.05            0.22       -0.12       -0.12        0.11       -0.21        0.19       -0.11        0.31
fast lr 0.02            0.54        0.42        0.20        0.46        0.40        0.65        0.49        0.81
fast lr 0.05            0.47        0.30        0.12        0.48        0.28        0.64        0.29        0.78
saved F:\forgetting\results_cifar\cifar_headline.png


In [5]:
DS = "cifar" if "cifar" in RESULTS_DIR else "mnist"

def run_mech(scenario, lr=0.05, seeds=3):
    global LR
    LR = lr; setup(scenario)
    pairs = [(a, b) for a in range(5) for b in range(5) if a != b]
    R = {k: [] for k in ["forget", "push_ratio", "gA_norm", "term_HBgA", "term_HAgB", "bracket",
                         "slope20", "curv20", "actual20", "slope_end", "curv_end", "actual_end"]}
    t0 = time.time()
    for seed in range(seeds):
        th0 = init(seed)
        th_task = {t: train(th0, t, seed) for t in range(5)}
        base_te = {t: evaluate(th_task[t], t)[1] for t in range(5)}
        base_est = {t: evaluate(th_task[t], t, "est")[1] for t in range(5)}
        gA0 = {t: grad(th_task[t], t) for t in range(5)}          # A's push when A finishes
        for a, b in pairs:
            thA = th_task[a]; log = []; at20 = {}
            def rec(step, th_, a=a, b=b, thA=thA, log=log, at20=at20):
                if step not in (5, 10, 15, 20): return
                gA, gB = grad(th_, a), grad(th_, b)
                t1, t2 = hvp(th_, b, gA), hvp(th_, a, gB)
                log.append((float(t1.norm()), float(t2.norm()), float((t1 - t2).norm())))
                if step == 20: at20["th"] = th_.detach().clone()
            thAB = train(thA, b, seed, record=rec)
            def split(th_):
                d = th_ - thA
                slope = float(gA0[a] @ d)
                curv = float(0.5 * d @ hvp(thA, a, d))
                actual = evaluate(th_, a, "est")[1] - base_est[a]
                return slope, curv, actual
            s20, c20, a20 = split(at20["th"]); se, ce, ae = split(thAB)
            R["forget"].append(evaluate(thAB, a)[1] - base_te[a])
            R["gA_norm"].append(float(gA0[a].norm()))
            R["push_ratio"].append(float(gA0[a].norm() / grad(thA, b).norm()))
            R["term_HBgA"].append(float(np.mean([x[0] for x in log])))
            R["term_HAgB"].append(float(np.mean([x[1] for x in log])))
            R["bracket"].append(float(np.mean([x[2] for x in log])))
            for k, v in zip(["slope20", "curv20", "actual20", "slope_end", "curv_end", "actual_end"], [s20, c20, a20, se, ce, ae]):
                R[k].append(v)
        print(f"  seed {seed} done after {time.time() - t0:.0f}s")
    r = lambda k: float(spearmanr(R[k], R["forget"])[0])
    share = lambda s, c: float(np.mean(np.abs(R[s])) / (np.mean(np.abs(R[s])) + np.mean(np.abs(R[c]))))
    Sm = {"dataset": DS, "scenario": scenario, "lr": lr, "seeds": seeds,
          "c1_push_A_mean": float(np.mean(R["gA_norm"])), "c1_push_ratio_A_over_B": float(np.mean(R["push_ratio"])),
          "c2_slope_share_step20": share("slope20", "curv20"), "c2_slope_share_end": share("slope_end", "curv_end"),
          "c2_taylor_vs_actual_step20": float(spearmanr(np.array(R["slope20"]) + np.array(R["curv20"]), R["actual20"])[0]),
          "c3_rho_full_bracket": r("bracket"), "c3_rho_HBgA": r("term_HBgA"), "c3_rho_HAgB": r("term_HAgB"),
          "c3_rho_slope20": r("slope20"), "c3_rho_curv20": r("curv20")}
    json.dump(Sm, open(os.path.join(RESULTS_DIR, f"mech_{DS}_{scenario}_lr{lr}.json"), "w"), indent=2)
    print(f"\n== MECH | {DS} | {'slow' if scenario == 'task' else 'fast'} | lr {lr}")
    for k, v in Sm.items():
        if k.startswith("c"): print(f"  {k:30s} {v:8.3f}")
    return Sm
print("defined")

defined


In [6]:
for sc in ["task", "domain"]:
    run_mech(sc, lr=0.05)

  seed 0 done after 72s
  seed 1 done after 141s
  seed 2 done after 209s

== MECH | cifar | slow | lr 0.05
  c1_push_A_mean                    0.791
  c1_push_ratio_A_over_B            0.937
  c2_slope_share_step20             0.568
  c2_slope_share_end                0.310
  c2_taylor_vs_actual_step20        0.994
  c3_rho_full_bracket               0.041
  c3_rho_HBgA                       0.044
  c3_rho_HAgB                       0.017
  c3_rho_slope20                    0.434
  c3_rho_curv20                     0.168
  seed 0 done after 63s
  seed 1 done after 126s
  seed 2 done after 190s

== MECH | cifar | fast | lr 0.05
  c1_push_A_mean                    0.795
  c1_push_ratio_A_over_B            0.384
  c2_slope_share_step20             0.229
  c2_slope_share_end                0.150
  c2_taylor_vs_actual_step20        0.876
  c3_rho_full_bracket               0.297
  c3_rho_HBgA                       0.253
  c3_rho_HAgB                       0.195
  c3_rho_slope20            

In [7]:
import glob
rows = [json.load(open(f)) for f in sorted(glob.glob(os.path.join(RESULTS_DIR, "mech_*.json")))]
keys = ["c1_push_A_mean", "c1_push_ratio_A_over_B", "c2_slope_share_step20", "c2_slope_share_end",
        "c2_taylor_vs_actual_step20", "c3_rho_full_bracket", "c3_rho_HAgB", "c3_rho_HBgA", "c3_rho_slope20", "c3_rho_curv20"]
print(f"{'':30s}" + "".join(f"{r['dataset'] + ' ' + ('slow' if r['scenario'] == 'task' else 'fast'):>14s}" for r in rows))
for k in keys:
    print(f"{k:30s}" + "".join(f"{r[k]:14.3f}" for r in rows))

                                  cifar fast    cifar slow
c1_push_A_mean                         0.795         0.791
c1_push_ratio_A_over_B                 0.384         0.937
c2_slope_share_step20                  0.229         0.568
c2_slope_share_end                     0.150         0.310
c2_taylor_vs_actual_step20             0.876         0.994
c3_rho_full_bracket                    0.297         0.041
c3_rho_HAgB                            0.195         0.017
c3_rho_HBgA                            0.253         0.044
c3_rho_slope20                         0.003         0.434
c3_rho_curv20                          0.732         0.168


In [5]:
from scipy.sparse.linalg import LinearOperator, eigsh
if "DS" not in globals(): DS = "cifar" if "cifar" in RESULTS_DIR else "mnist"

def curv_shape(scenario, lr=0.05, seeds=2):
    global LR
    LR = lr; setup(scenario)
    lmax, lmin, cosv = [], [], []; t0 = time.time()
    for seed in range(seeds):
        th0 = init(seed)
        th_task = {t: train(th0, t, seed) for t in range(5)}
        for a in range(5):
            th = th_task[a]
            def mv(v, th=th, a=a):
                v_t = torch.as_tensor(np.asarray(v).ravel(), dtype=torch.float32, device=dev)
                return hvp(th, a, v_t).double().cpu().numpy()
            op = LinearOperator((NPARAM, NPARAM), matvec=mv, dtype=np.float64)
            lmax.append(float(eigsh(op, k=1, which="LA", tol=1e-3, return_eigenvectors=False).max()))
            lmin.append(float(eigsh(op, k=1, which="SA", tol=1e-3, return_eigenvectors=False).min()))
            for b in range(5):
                if b != a:
                    gB = grad(th, b); cosv.append(float(F.cosine_similarity(gB, hvp(th, a, gB), dim=0)))
        print(f"  seed {seed} done after {time.time() - t0:.0f}s")
    ratio = [abs(min(mn, 0)) / mx for mn, mx in zip(lmin, lmax)]
    Sm = {"dataset": DS, "scenario": scenario, "lr": lr, "seeds": seeds,
          "largest_curvature_mean": float(np.mean(lmax)), "most_negative_mean": float(np.mean(lmin)),
          "neg_to_pos_ratio": float(np.mean(ratio)), "cos_gB_HAgB_mean": float(np.mean(cosv)),
          "cos_below_0.3_share": float(np.mean(np.array(cosv) < 0.3))}
    json.dump(Sm, open(os.path.join(RESULTS_DIR, f"curv_{DS}_{scenario}_lr{lr}.json"), "w"), indent=2)
    print(f"\n== VALLEY | {DS} | {'slow' if scenario == 'task' else 'fast'} | lr {lr}")
    for k, v in Sm.items():
        if k not in ("dataset", "scenario", "lr", "seeds"): print(f"  {k:24s} {v:9.4f}")
    return Sm

for sc in ["task", "domain"]:
    curv_shape(sc, lr=0.05)

  seed 0 done after 23s
  seed 1 done after 46s

== VALLEY | cifar | slow | lr 0.05
  largest_curvature_mean     16.1720
  most_negative_mean         -2.2506
  neg_to_pos_ratio            0.1531
  cos_gB_HAgB_mean            0.1107
  cos_below_0.3_share         0.9500
  seed 0 done after 20s
  seed 1 done after 42s

== VALLEY | cifar | fast | lr 0.05
  largest_curvature_mean     16.2593
  most_negative_mean         -2.2120
  neg_to_pos_ratio            0.1496
  cos_gB_HAgB_mean            0.8222
  cos_below_0.3_share         0.0000
